In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

PATH_MERGED = "data/to_use/merged.csv"
PATH_SURVEY = "data/to_use/survey_features_wide.csv"
PATH_OUT    = "data/to_use/panel_model.csv"

PID       = "Participant.Unique.ID"
KEYS      = [PID, "Year"]
INDEX_YRS = (2022, 2024)     # 2025 has no observable t+1
HISTORY_FLOOR = 2019
AGE_CAP   = 24               # verify whether cap binds at application or in-program

df = pd.read_csv(PATH_MERGED, low_memory=False)
df = df[df[PID].notna()].copy()

print(f"{len(df):,} rows | {df[PID].nunique():,} people | years {sorted(df.Year.unique())}")
print(f"\nservice_option:\n{df.service_option.value_counts().to_string()}")

/Applications/Positron.app/Contents/Resources/app/extensions/positron-python/python_files/lib/ipykernel/py3/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/sonia/Documents/SYEP
868,393 rows | 443,410 people | years [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

service_option:
service_option
Older Youth            611154
Younger Youth          208035
Ladders for Leaders     49204


In [3]:
df["_enr"] = df["enrolled"].fillna(False).astype(bool)

py = (df.groupby(KEYS, as_index=False)
        .agg(applied=("application_id", "size"),
             enrolled_any=("_enr", "max")))
py["applied"] = 1
py["enrolled_any"] = py["enrolled_any"].astype(int)

print(f"person-years: {len(py):,}")
print(py.groupby("Year")[["applied", "enrolled_any"]].sum().to_string())

person-years: 836,035
      applied  enrolled_any
Year                       
2019   126442         56891
2020    86344             0
2021    93852         40955
2022   121043         60174
2023   124443         62614
2024   131772         63965
2025   152139         64660


## 2. Prior history

Counts of *distinct prior years* with an application / enrollment, strictly before the
index year, floored at 2019. `shift()` after `cumsum()` makes these strictly prior —
off-by-one here would leak the current year into its own predictor.

In [4]:
py = py.sort_values(KEYS).reset_index(drop=True)
g = py.groupby(PID, sort=False)

py["n_prior_apps"]   = (g["applied"].cumsum() - py["applied"]).astype(int)
py["n_prior_enroll"] = (g["enrolled_any"].cumsum() - py["enrolled_any"]).astype(int)

py["first_year"] = g["Year"].transform("min")
py["prev_year"]  = g["Year"].shift(1)
py["gap"]        = py["Year"] - py["prev_year"]

prev_enr = g["enrolled_any"].shift(1)
py["prev_enrolled"] = prev_enr

# Sanity: priors must be 0 in the floor year, and cannot exceed elapsed years.
assert (py.loc[py.Year == HISTORY_FLOOR, "n_prior_apps"] == 0).all()
assert (py["n_prior_apps"] <= py["Year"] - HISTORY_FLOOR).all()

print(pd.crosstab(py.Year, py.n_prior_apps).to_string())

n_prior_apps       0      1      2      3     4     5     6
Year                                                       
2019          126442      0      0      0     0     0     0
2020           35977  50367      0      0     0     0     0
2021           50444  23863  19545      0     0     0     0
2022           60708  31522  18597  10216     0     0     0
2023           53454  34679  19532  11019  5759     0     0
2024           53933  32944  22578  12361  6550  3406     0
2025           62452  36520  23686  15496  7970  4039  1976


## 3. Forward outcome

Shift `Year` down by one so each row carries what happened the *following* year.

In [5]:
nxt = (py[KEYS + ["applied", "enrolled_any"]]
         .assign(Year=lambda d: d.Year - 1)
         .rename(columns={"applied": "reapplied_next",
                          "enrolled_any": "reenrolled_next"}))

py = py.merge(nxt, on=KEYS, how="left", validate="1:1")
py[["reapplied_next", "reenrolled_next"]] = py[["reapplied_next", "reenrolled_next"]].fillna(0).astype(int)

print(py.groupby("Year")[["reapplied_next", "reenrolled_next"]].mean().round(3).to_string())

      reapplied_next  reenrolled_next
Year                                 
2019           0.398            0.000
2020           0.354            0.175
2021           0.437            0.243
2022           0.465            0.266
2023           0.492            0.267
2024           0.517            0.253
2025           0.000            0.000


In [ ]:
# 2020 is zero because of issues with enrollment status

## 4. Collapse attributes to person-year

32,363 person-years have 2 application rows (max 2, so no deeper nesting). Prefer the
enrolled row; break remaining ties on `total_hours_paid` descending. Hours are summed
across rows, since two placements mean two sets of hours worked.

In [6]:
ATTRS = ["service_option", "program_type", "Cohort", "age_on_start", "app_status",
         "outcome", "provider", "organization", "borough", "worksite_id",
         "Gender", "Race.Ethnicity", "Educational.Status", "Work.Status",
         "Previous.Work.Experience", "NYCHA.Housing", "Public.Assistance",
         "Citizenship.Status", "English.Proficiency"]
ATTRS = [c for c in ATTRS if c in df.columns]

d = df.sort_values(KEYS + ["_enr", "total_hours_paid"])
attrs = d.groupby(KEYS, as_index=False)[ATTRS].last()

SUM_COLS = [c for c in ["total_hours_paid", "Training.Hours", "Hours.Worked"] if c in df.columns]
sums = df.groupby(KEYS, as_index=False)[SUM_COLS].sum(min_count=1)

panel = (py.merge(attrs, on=KEYS, how="left", validate="1:1")
           .merge(sums,  on=KEYS, how="left", validate="1:1"))

assert panel.duplicated(KEYS).sum() == 0
print(f"panel: {len(panel):,} person-years")
print(f"\nenrolled_any (person-year): {panel.enrolled_any.mean():.3f}")

panel: 836,035 person-years

enrolled_any (person-year): 0.418


## 5. Filter to the analysis frame

In [7]:
frame = panel[(panel.service_option == "Older Youth") &
              (panel.Year.between(*INDEX_YRS))].copy()
print(f"Older Youth, index years: {len(frame):,} rows | {frame[PID].nunique():,} people")
print(frame.Year.value_counts().sort_index().to_string())

# Age eligibility: only drop those who definitively cannot reapply.
print(f"\nage distribution:\n{frame.age_on_start.value_counts().sort_index().to_string()}")
frame = frame[frame.age_on_start < AGE_CAP].copy()
print(f"\nafter age filter: {len(frame):,}")

# Complete history = turned 16 in 2019 or later, so priors are not left-censored.
frame["age_in_2019"] = frame.age_on_start - (frame.Year - HISTORY_FLOOR)
frame["complete_history"] = frame.age_in_2019 <= 16
print(f"complete history: {frame.complete_history.mean():.3f}")

Older Youth, index years: 261,620 rows | 182,381 people
Year
2022    83292
2023    85891
2024    92437

age distribution:
age_on_start
14.0       15
15.0       36
16.0    58504
17.0    55714
18.0    43748
19.0    33914
20.0    24886
21.0    17831
22.0    12826
23.0     8735
24.0     5403
25.0        6
28.0        1

after age filter: 256,209
complete history: 0.836


In [8]:
frame = (frame.sort_values(KEYS)
              .groupby(PID, as_index=False)
              .first())
print(f"first-year only: {len(frame):,} rows")
print(frame.Year.value_counts().sort_index().to_string())
print(f"\nreapplied_next: {frame.reapplied_next.mean():.3f} "
      f"({frame.reapplied_next.sum():,} events)")

first-year only: 179,282 rows
Year
2022    81865
2023    49893
2024    47524

reapplied_next: 0.445 (79,802 events)


## 6. Derived predictors

In [9]:
frame["any_prior_app"] = (frame.n_prior_apps > 0).astype(int)
frame["prior_enroll_rate"] = np.where(frame.n_prior_apps > 0,
                                      frame.n_prior_enroll / frame.n_prior_apps, np.nan)
H = "total_hours_paid"
print(
    frame.groupby("Year")[H].agg(
        mean="mean",
        std="std",
        nunique="nunique",
        missing_rate=lambda s: s.isna().mean()
    ).round(2).to_string()
)

       mean    std  nunique  missing_rate
Year                                     
2022  53.14  66.72      568           0.0
2023  47.43  64.59      528           0.0
2024  46.88  65.56      513           0.0


In [10]:
if frame[H].nunique() > 5:
    frame["hours_q"] = pd.qcut(frame[H], 5, labels=False, duplicates="drop")
    emp = frame.groupby("hours_q").agg(n=(H, "size"), mid=(H, "median"),
                                       rate=("reapplied_next", "mean"))
    emp["logit"] = np.log(emp.rate.clip(1e-6, 1 - 1e-6) / (1 - emp.rate.clip(1e-6, 1 - 1e-6)))
    print(emp.round(3).to_string())

              n    mid   rate  logit
hours_q                             
0        143748    0.0  0.388 -0.458
1         35534  150.0  0.678  0.745


## 7. Site-level feasibility

In [13]:
for c in ["provider", "organization", "worksite_id"]:
    if c not in frame.columns:
        continue
    sz = frame[c].value_counts()
    print(f"{c}: {frame[c].nunique():,} levels | {frame[c].isna().mean():.1%} null | "
          f"median size {sz.median():.0f} | {(sz < 20).sum():,} levels with <20 rows")

provider: 48 levels | 9.3% null | median size 2677 | 1 levels with <20 rows
organization: 51 levels | 9.3% null | median size 2584 | 1 levels with <20 rows
worksite_id: 16,579 levels | 46.7% null | median size 3 | 15,654 levels with <20 rows


In [14]:
MIN_N = 20
keep = frame.provider.value_counts()
keep = keep[keep >= MIN_N].index
frame["provider_c"] = np.where(frame.provider.isin(keep), frame.provider, "Other")
print(f"provider_c: {frame.provider_c.nunique()} levels "
      f"({(frame.provider_c == 'Other').sum():,} rows pooled into Other)")

provider_c: 48 levels (16,617 rows pooled into Other)


## 8. Merge survey features

In [15]:
wide = pd.read_csv(PATH_SURVEY)
print(f"survey frame: {len(wide):,} rows, years {sorted(wide.Year.unique())}")

panel_model = frame.merge(wide, on=KEYS, how="left", validate="1:1", indicator="_m")
panel_model["has_survey"] = panel_model._m.eq("both")
panel_model = panel_model.drop(columns="_m")

srv = panel_model[panel_model.has_survey]
print(f"\nadmin frame:      {len(panel_model):,} rows | {panel_model.reapplied_next.sum():,} events")
print(f"survey subsample: {len(srv):,} rows | {srv.reapplied_next.sum():,} events")

ev = int(srv.reapplied_next.sum())

survey frame: 8,882 rows, years [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

admin frame:      179,282 rows | 79,802 events
survey subsample: 5,204 rows | 3,609 events


In [16]:
# Are survey responders representative?
cmp_cols = [c for c in ["reapplied_next", "reenrolled_next", "age_on_start",
                        "n_prior_apps", "n_prior_enroll", "total_hours_paid",
                        "enrolled_any", "complete_history"]
            if c in panel_model.columns]
print(panel_model.groupby("has_survey")[cmp_cols].mean().round(3).T.to_string())

has_survey         False    True 
reapplied_next     0.438    0.694
reenrolled_next    0.235    0.428
age_on_start      17.932   17.586
n_prior_apps       0.840    0.817
n_prior_enroll     0.340    0.327
total_hours_paid  47.407  133.027
enrolled_any       0.460    1.000
complete_history   0.830    0.859


In [17]:
panel_model.to_csv(PATH_OUT, index=False)
print(f"wrote {PATH_OUT}: {panel_model.shape}")

wrote data/to_use/panel_model.csv: (179282, 48)


Participants with matched pre- and post-survey responses reapply at 69%, compared to 44% among those with only one wave or no survey — a 25-point gap. Most of it is structural rather than behavioral.

Responding requires enrollment. Survey responders are 100% enrolled; non-responders are 46%. The post-survey is administered during the program, so the majority of the comparison group — applicants who were declined, not selected, or selected but never enrolled — cannot appear in the matched sample by construction. 

The comparison is therefore not responders vs. non-responders so much as enrollees vs. the full applicant pool.

The hours gap follows from that. Mean paid hours are 133 among responders versus 47 among non-responders. Since 54% of non-responders never enrolled and have zero hours by construction, this reflects the enrollment composition of the group rather than shorter or interrupted placements among those who did participate.

Participation history is balanced. Prior applications (0.82 vs. 0.84), prior enrollments (0.33 vs. 0.34), age (17.6 vs. 17.9), and complete-history coverage (86% vs. 83%) are near-identical across the two groups.

Both groups reapply at roughly double their re-enrollment rate. Responders reapply at 69% but re-enroll at 43%; non-responders at 44% and 24%. The ~20-point wedge in both groups is allocation, not preference — participants want to return considerably more often than capacity allows. This is the direct justification for using reapplication rather than re-enrollment as the outcome: re-enrollment would measure the lottery as much as the participant.
